In [1]:
import sys
sys.path.append("../../")

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from NEW_PINN.PINN_const import EINN_PINN


# ============================================
# CONFIG
# ============================================

DATA_PATH = "../synthetic_datasets/01_baseline_constant.csv"

SAVE_DIR = "preliminary_experiments/Baseline_PINN_Results/"

train_size = 150
test_size = 30
n_trials = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRUE_PARAMS = {
    "beta": 0.1,
    "gamma": 0.06,
    "mu": 0.003
}

os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================
# SEED
# ============================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================
# ADD NOISE
# ============================================

def add_noise_to_dataset(df, noise_level=0.05, seed=42):

    np.random.seed(seed)
    df_noisy = df.copy()

    I_noise = np.random.normal(
        0,
        noise_level * df['I'].max(),
        len(df)
    )

    df_noisy['I'] = np.maximum(
        0,
        df['I'] + I_noise
    )

    D_noise = np.random.normal(
        0,
        noise_level * df['D'].max(),
        len(df)
    )

    df_noisy['D'] = np.maximum(
        0,
        df['D'] + D_noise
    )

    return df_noisy


# ============================================
# METRICS
# ============================================

def compute_metrics(true, pred):

    rmse = np.sqrt(np.mean((true - pred) ** 2))
    mae = np.mean(np.abs(true - pred))

    return rmse, mae


def relative_error(true, pred):

    return np.abs(pred - true) / true


def compute_peak_errors(true_I, pred_I):

    true_peak_idx = np.argmax(true_I)
    pred_peak_idx = np.argmax(pred_I)

    true_peak = true_I[true_peak_idx]
    pred_peak = pred_I[pred_peak_idx]

    peak_height_error = np.abs(
        pred_peak - true_peak
    ) / true_peak * 100

    peak_timing_error = np.abs(
        pred_peak_idx - true_peak_idx
    )

    return peak_height_error, peak_timing_error


# ============================================
# DATASETS
# ============================================

def load_datasets():

    df = pd.read_csv(DATA_PATH)

    dataset_1 = df.copy()

    dataset_2 = add_noise_to_dataset(df, 0.03)
    dataset_3 = add_noise_to_dataset(df, 0.05)

    return {
        "synt_dataset_1": dataset_1,
        "synt_dataset_2": dataset_2,
        "synt_dataset_3": dataset_3
    }


# ============================================
# PLOTS
# ============================================

def plot_I_predictions(dataset_name, df, predictions):

    t = df["day"].values
    I_true = df["I"].values

    plt.figure(figsize=(10,6))

    for pred in predictions:
        plt.plot(t, pred, color="blue", alpha=0.3)

    plt.plot(t, I_true, color="black", linewidth=2, label="True")

    plt.axvline(
        train_size,
        linestyle="--",
        color="red",
        label="Train/Test split"
    )

    plt.title(f"I predictions — {dataset_name}")
    plt.xlabel("Days")
    plt.ylabel("I")
    plt.legend()

    plt.savefig(
        os.path.join(
            SAVE_DIR,
            f"s_d_I_predictions_{dataset_name}.png"
        )
    )

    plt.close()


def plot_param_boxplot(dataset_name, results_df):

    plt.figure(figsize=(8,6))

    data = [
        results_df["beta_error"],
        results_df["gamma_error"],
        results_df["mu_error"]
    ]

    plt.boxplot(data)

    plt.xticks(
        [1,2,3],
        ["beta","gamma","mu"]
    )

    plt.title(f"Parameter Errors — {dataset_name}")

    plt.savefig(
        os.path.join(
            SAVE_DIR,
            f"s_d_param_boxplot_{dataset_name}.png"
        )
    )

    plt.close()


def plot_param_trials(dataset_name, results_df):

    plt.figure(figsize=(10,6))

    plt.plot(
        results_df["trial"],
        results_df["beta_error"],
        label="beta"
    )

    plt.plot(
        results_df["trial"],
        results_df["gamma_error"],
        label="gamma"
    )

    plt.plot(
        results_df["trial"],
        results_df["mu_error"],
        label="mu"
    )

    plt.legend()

    plt.title(f"Parameter Errors per Trial — {dataset_name}")
    plt.xlabel("Trial")
    plt.ylabel("Relative Error")

    plt.savefig(
        os.path.join(
            SAVE_DIR,
            f"s_d_param_trials_{dataset_name}.png"
        )
    )

    plt.close()

# ============================================
# TRAIN SINGLE TRIAL
# ============================================

def run_single_trial(df, seed):

    set_seed(seed)

    t = df["day"].values

    S = df["S"].values
    I = df["I"].values
    R = df["R"].values
    D = df["D"].values

    population = S[0] + I[0] + R[0] + D[0]

    model = EINN_PINN(
        t,
        S,
        I,
        R,
        D,
        population,
        train_size=train_size,
        device=device
    ).to(device)

    model.train_model(
        n_epoch=20_000
    )

    S_pred, I_pred, R_pred, D_pred = model.predict()

    S_pred = S_pred.numpy()
    I_pred = I_pred.numpy()
    R_pred = R_pred.numpy()
    D_pred = D_pred.numpy()

    # test range
    test_slice = slice(
        train_size,
        train_size + test_size
    )

    I_true_test = I[test_slice]
    I_pred_test = I_pred[test_slice]

    # metrics
    rmse, mae = compute_metrics(
        I_true_test,
        I_pred_test
    )

    peak_height, peak_timing = compute_peak_errors(
        I_true_test,
        I_pred_test
    )

    params = model.params.get_params_dict()

    beta_error = relative_error(
        TRUE_PARAMS["beta"],
        params["beta"]
    )

    gamma_error = relative_error(
        TRUE_PARAMS["gamma"],
        params["gamma"]
    )

    mu_error = relative_error(
        TRUE_PARAMS["mu"],
        params["mu"]
    )

    return {
        "RMSE": rmse,
        "MAE": mae,
        "beta_error": beta_error,
        "gamma_error": gamma_error,
        "mu_error": mu_error,
        "peak_height": peak_height,
        "peak_timing": peak_timing,
        "I_pred": I_pred
    }


# ============================================
# RUN EXPERIMENT
# ============================================

def run_experiment():

    datasets = load_datasets()
    all_predictions = []
    summary_rows = []

    for dataset_name, df in datasets.items():

        print(f"\nRunning {dataset_name}")

        results = []

        for trial in range(n_trials):

            seed = 42 + trial

            print(f"Trial {trial}")

            result = run_single_trial(
                df,
                seed
            )

            result["trial"] = trial
            results.append(result)
            all_predictions.append(result["I_pred"]) 

        results_df = pd.DataFrame(results)

        results_df.to_csv(
            os.path.join(
                SAVE_DIR,
                f"s_d_{dataset_name}.csv"
            ),
            index=False
        )
        plot_I_predictions(
            dataset_name,
            df,
            all_predictions
        )

        plot_param_boxplot(
            dataset_name,
            results_df
        )

        plot_param_trials(
            dataset_name,
            results_df
        )
        summary = {

            "dataset": dataset_name,

            "I RMSE (mean ± std)":
                f"{results_df['RMSE'].mean():.4f} ± {results_df['RMSE'].std():.4f}",

            "I MAE (mean ± std)":
                f"{results_df['MAE'].mean():.4f} ± {results_df['MAE'].std():.4f}",

            "beta Relative Error (mean ± std)":
                f"{results_df['beta_error'].mean():.4f} ± {results_df['beta_error'].std():.4f}",

            "gamma Relative Error (mean ± std)":
                f"{results_df['gamma_error'].mean():.4f} ± {results_df['gamma_error'].std():.4f}",

            "mu Relative Error (mean ± std)":
                f"{results_df['mu_error'].mean():.4f} ± {results_df['mu_error'].std():.4f}",

            "Peak Height Error (%)":
                f"{results_df['peak_height'].mean():.4f} ± {results_df['peak_height'].std():.4f}",

            "Peak Timing Error (days)":
                f"{results_df['peak_timing'].mean():.4f} ± {results_df['peak_timing'].std():.4f}"

        }

        summary_rows.append(summary)

    summary_df = pd.DataFrame(summary_rows)

    summary_df.to_csv(
        os.path.join(
            SAVE_DIR,
            "s_d_summary_table.csv"
        ),
        index=False
    )

    print("\nFinished experiments")
    print(summary_df)


# ============================================
# RUN
# ============================================

if __name__ == "__main__":

    run_experiment()


Running synt_dataset_1
Trial 0


c:\Users\dinara\Desktop\НИР\Code\PINN_LLM\venv\Lib\site-packages\torch\optim\lr_scheduler.py:1340: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  current = float(metrics)


Epoch     0 | Loss: 13342.279297 | Data: 12450.599609 | ODE: 69.298264
Params: β=0.3000, γ=0.1000, μ=0.0100
---
Epoch  1000 | Loss: 434.707123 | Data: 379.746857 | ODE: 17.103920
Params: β=0.2670, γ=0.0997, μ=0.0118
---
Epoch  2000 | Loss: 79.310387 | Data: 76.686668 | ODE: 15.322023
Params: β=0.2419, γ=0.0953, μ=0.0138
---
Epoch  3000 | Loss: 37.240040 | Data: 35.875099 | ODE: 11.032721
Params: β=0.2144, γ=0.0822, μ=0.0161
---
Epoch  4000 | Loss: 33.534859 | Data: 32.560261 | ODE: 7.579321
Params: β=0.1903, γ=0.0720, μ=0.0178
---
Epoch  5000 | Loss: 70.202141 | Data: 69.470345 | ODE: 5.281314
Params: β=0.1697, γ=0.0646, μ=0.0177
---
Epoch  6000 | Loss: 18.559361 | Data: 18.119532 | ODE: 3.289971
Params: β=0.1518, γ=0.0600, μ=0.0158
---
Epoch  7000 | Loss: 4.086794 | Data: 3.776067 | ODE: 2.079908
Params: β=0.1370, γ=0.0576, μ=0.0135
---
Epoch  8000 | Loss: 3.421266 | Data: 3.180000 | ODE: 1.381549
Params: β=0.1252, γ=0.0552, μ=0.0114
---
Epoch  9000 | Loss: 13.043365 | Data: 12.840195

In [1]:
import os
import pandas as pd
import numpy as np

SAVE_DIR = "preliminary_experiments/Baseline_PINN_Results/"

files = [
    "s_d_synt_dataset_1.csv",
    "s_d_synt_dataset_2.csv",
    "s_d_synt_dataset_3.csv"
]

summary_rows = []

for file in files:

    path = os.path.join(SAVE_DIR, file)

    df = pd.read_csv(path)

    dataset_name = file.replace("s_d_", "").replace(".csv", "")

    summary = {

        "dataset": dataset_name,

        "I RMSE (mean ± std)":
            f"{df['RMSE'].mean():.4f} ± {df['RMSE'].std():.4f}",

        "I MAE (mean ± std)":
            f"{df['MAE'].mean():.4f} ± {df['MAE'].std():.4f}",

        "beta Relative Error (mean ± std)":
            f"{df['beta_error'].mean():.4f} ± {df['beta_error'].std():.4f}",

        "gamma Relative Error (mean ± std)":
            f"{df['gamma_error'].mean():.4f} ± {df['gamma_error'].std():.4f}",

        "mu Relative Error (mean ± std)":
            f"{df['mu_error'].mean():.4f} ± {df['mu_error'].std():.4f}",

        "Peak Height Error (%)":
            f"{df['peak_height'].mean():.4f} ± {df['peak_height'].std():.4f}",

        "Peak Timing Error (days)":
            f"{df['peak_timing'].mean():.4f} ± {df['peak_timing'].std():.4f}"

    }

    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "s_d_summary_table_updated.csv"
    ),
    index=False
)

print(summary_df)

          dataset I RMSE (mean ± std) I MAE (mean ± std)  \
0  synt_dataset_1     3.8281 ± 2.5204    3.2575 ± 2.1142   
1  synt_dataset_2     4.6098 ± 2.0049    3.7018 ± 1.6573   
2  synt_dataset_3     6.1055 ± 1.9181    4.8717 ± 1.5703   

  beta Relative Error (mean ± std) gamma Relative Error (mean ± std)  \
0                  0.0225 ± 0.0143                   0.0927 ± 0.0265   
1                  0.0212 ± 0.0158                   0.0956 ± 0.0256   
2                  0.0283 ± 0.0165                   0.0993 ± 0.0357   

  mu Relative Error (mean ± std) Peak Height Error (%)  \
0                0.8391 ± 0.1175       1.9921 ± 0.9935   
1                0.8216 ± 0.1520       6.1042 ± 1.0194   
2                0.8176 ± 0.1526       9.4103 ± 1.1008   

  Peak Timing Error (days)  
0          4.2000 ± 2.5298  
1          2.3000 ± 1.7029  
2          2.3000 ± 2.0575  


In [2]:
import os
import pandas as pd
import numpy as np

SAVE_DIR = "preliminary_experiments/Baseline_PINN_Results/"

TRUE_PARAMS = {
    "beta": 0.1,
    "gamma": 0.06,
    "mu": 0.003
}

files = [
    "s_d_synt_dataset_1.csv",
    "s_d_synt_dataset_2.csv",
    "s_d_synt_dataset_3.csv"
]

rows = []

for file in files:

    df = pd.read_csv(os.path.join(SAVE_DIR, file))

    dataset = file.replace("s_d_", "").replace(".csv", "")

    # восстановление predicted
    beta_pred = TRUE_PARAMS["beta"] * (1 + df["beta_error"])
    gamma_pred = TRUE_PARAMS["gamma"] * (1 + df["gamma_error"])
    mu_pred = TRUE_PARAMS["mu"] * (1 + df["mu_error"])

    row = {

        "Dataset": dataset,

        "I RMSE (mean ± std)":
            f"{df['RMSE'].mean():.4f} ± {df['RMSE'].std():.4f}",

        "I MAE (mean ± std)":
            f"{df['MAE'].mean():.4f} ± {df['MAE'].std():.4f}",

        "β real": TRUE_PARAMS["beta"],
        "γ real": TRUE_PARAMS["gamma"],
        "μ real": TRUE_PARAMS["mu"],

        "β predicted (mean ± std)":
            f"{beta_pred.mean():.4f} ± {beta_pred.std():.4f}",

        "γ predicted (mean ± std)":
            f"{gamma_pred.mean():.4f} ± {gamma_pred.std():.4f}",

        "μ predicted (mean ± std)":
            f"{mu_pred.mean():.6f} ± {mu_pred.std():.6f}"

    }

    rows.append(row)

summary = pd.DataFrame(rows)

print(summary)

summary.to_csv(
    os.path.join(
        SAVE_DIR,
        "s_d_param_summary.csv"
    ),
    index=False
)

          Dataset I RMSE (mean ± std) I MAE (mean ± std)  β real  γ real  \
0  synt_dataset_1     3.8281 ± 2.5204    3.2575 ± 2.1142     0.1    0.06   
1  synt_dataset_2     4.6098 ± 2.0049    3.7018 ± 1.6573     0.1    0.06   
2  synt_dataset_3     6.1055 ± 1.9181    4.8717 ± 1.5703     0.1    0.06   

   μ real β predicted (mean ± std) γ predicted (mean ± std)  \
0   0.003          0.1022 ± 0.0014          0.0656 ± 0.0016   
1   0.003          0.1021 ± 0.0016          0.0657 ± 0.0015   
2   0.003          0.1028 ± 0.0017          0.0660 ± 0.0021   

  μ predicted (mean ± std)  
0      0.005517 ± 0.000353  
1      0.005465 ± 0.000456  
2      0.005453 ± 0.000458  
